In [ ]:
import warnings
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import warnings; warnings.simplefilter('ignore')
from scipy.spatial import cKDTree
from shapely.geometry import Point, Polygon as ShapelyPolygon
from shapely.ops import unary_union
from shapely.strtree import STRtree

from esda.moran import Moran_Local
from libpysal.weights import DistanceBand
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, RegularPolygon
from matplotlib.collections import PatchCollection
from scipy.spatial import cKDTree
from esda.moran import Moran_Local_BV
from libpysal.weights import DistanceBand

from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})


plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


output_dir = str(P.results.figures / "figure_7")
os.makedirs(output_dir, exist_ok=True)


# setup

In [ ]:

INPUT_ADATA_PATH = str(P.processed.adata.all_cells / P.fn.all_cells_final)

adata = sc.read_h5ad(INPUT_ADATA_PATH)
run_name = "morans25"

# ─── Column / label constants ───────────────────────────────────────────────
CORE_COL         = 'core_id'
CELLTYPE_COL     = 'annotation_final_fine_cd8'
TISSUE_COL       = 'tissue_type_cell_level_normal_split'
PATIENT_COL      = 'patient_id'

EPITHELIAL_LABEL = 'Epithelial'
CD8_LABEL        = 'CD8-T-Cell'
TCELL_LABEL      = 'T-Cells'
GDF15_GENE       = 'GDF15'
CD8A_GENE        = 'CD8A'
CD3E_GENE        = 'CD3E'

In [ ]:
epi_mask = adata.obs[CELLTYPE_COL] == EPITHELIAL_LABEL
epi_obs = adata.obs.loc[epi_mask, [CORE_COL, TISSUE_COL]].copy()

epi_obs[TISSUE_COL] = epi_obs[TISSUE_COL].astype(str)
epi_obs = epi_obs[
    ~epi_obs[TISSUE_COL].isin(['nan', 'NA', 'NaN', 'None', ''])
]

def _resolve(group):
    uniq = group.unique()
    if len(uniq) == 0:
        return np.nan
    if len(uniq) == 1:
        return uniq[0]
    return 'Mixed'

core_to_tissue = (
    epi_obs.groupby(CORE_COL, observed=True)[TISSUE_COL]
           .apply(_resolve)
)

adata.obs['core_level_tissue_type'] = (
    adata.obs[CORE_COL].map(core_to_tissue)
)

print(adata.obs['core_level_tissue_type'].value_counts(dropna=False))
print("\nCores per resolved tissue:")
print(core_to_tissue.value_counts(dropna=False))

# nearest epi 30 morans

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import PatchCollection
from esda.moran import Moran_Local_BV
from libpysal.weights import DistanceBand


# ─── Column / label constants ───────────────────────────────────────────────
CORE_COL         = 'core_id'
CELLTYPE_COL     = 'annotation_final_fine_cd8'
TISSUE_COL       = 'tissue_type_cell_level_normal_split'
CORE_TISSUE_COL  = 'core_level_tissue_type'
PATIENT_COL      = 'patient_id'

EPITHELIAL_LABEL = 'Epithelial'
CD8_LABEL        = 'CD8-T-Cell'
TCELL_LABEL      = 'T-Cells'
GDF15_GENE       = 'GDF15'
CD8A_GENE        = 'CD8A'
CD3E_GENE        = 'CD3E'


# ─── Pipeline config ────────────────────────────────────────────────────────
BIN_SIZE_UM      = 30
MIN_EPI_PER_BIN  = 1
MIN_EPI_FRACTION = 0.0
N_PERMUTATIONS   = 999
LISA_SEED        = 31

BV_QUADRANT = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
QUADRANT_COLORS = {
    'HH':       '#f5012d',
    'HL':       '#ff7933',
    'LH':       '#a30180',
    'LL':       '#000080',
    'NS':       '#adcadb',
    'NOT_RUN':  'none',
}


# ─── Aggregation: all cells, drop only empty bins ──────────────────────────
def aggregate_to_squarebins_all_cells(
    adata, bin_size_um=BIN_SIZE_UM,
    gdf15_gene=GDF15_GENE,
    cd8a_gene=CD8A_GENE,
    cd3e_gene=CD3E_GENE,
):
    obs = adata.obs
    coords = adata.obsm['spatial']
    core_arr   = obs[CORE_COL].values
    ct_arr     = obs[CELLTYPE_COL].values
    is_epi     = ct_arr == EPITHELIAL_LABEL
    is_cd8     = ct_arr == CD8_LABEL
    tissue_arr = obs[TISSUE_COL].astype(str).values

    if CORE_TISSUE_COL not in obs.columns:
        raise ValueError(
            f"adata.obs is missing {CORE_TISSUE_COL!r}. "
            f"Build it before running aggregation."
        )
    core_tissue_map = (
        obs[[CORE_COL, CORE_TISSUE_COL]]
        .drop_duplicates()
        .set_index(CORE_COL)[CORE_TISSUE_COL]
    )

    def _pull(g):
        if g not in adata.var_names:
            raise ValueError(f"{g!r} not in adata.var_names")
        v = adata[:, g].layers['counts'] if 'counts' in adata.layers else adata[:, g].X
        v = v.toarray().flatten() if hasattr(v, 'toarray') else np.asarray(v).flatten()
        return v.astype(np.float64)

    gdf15 = _pull(gdf15_gene)
    cd8a  = _pull(cd8a_gene)
    cd3e  = _pull(cd3e_gene)

    rows = []
    bin_id_per_cell = np.full(adata.n_obs, '', dtype=object)
    bin_geometry = {}

    n_mixed_cores_seen = 0
    n_mixed_bins_total = 0
    n_mixed_bins_epi = 0
    n_mixed_bins_inherited = 0
    n_mixed_bins_no_epi_in_core = 0

    for core_id in np.unique(core_arr):
        sel = core_arr == core_id
        idxs = np.flatnonzero(sel)
        if len(idxs) < 5:
            continue
        cc = coords[idxs]

        core_label = core_tissue_map.get(core_id, 'Unknown')
        if pd.isna(core_label):
            core_label = 'Unknown'
        core_label = str(core_label)
        is_mixed_core = (core_label == 'Mixed')

        ix = np.floor(cc[:, 0] / bin_size_um).astype(np.int64)
        iy = np.floor(cc[:, 1] / bin_size_um).astype(np.int64)
        bin_keys = pd.Series(list(zip(ix, iy)))
        bin_idx_per_cell, unique_keys = pd.factorize(bin_keys)
        unique_ix = np.array([k[0] for k in unique_keys])
        unique_iy = np.array([k[1] for k in unique_keys])

        center_x = (unique_ix + 0.5) * bin_size_um
        center_y = (unique_iy + 0.5) * bin_size_um

        bin_geometry[core_id] = {
            'bin_size_um': bin_size_um,
            'extent':      [cc[:, 0].min(), cc[:, 0].max(),
                            cc[:, 1].min(), cc[:, 1].max()],
        }

        n_bins = len(unique_keys)
        bin_records = []         
        bin_tissue_provisional = []   
        bin_has_epi = np.zeros(n_bins, dtype=bool)

        for bi in range(n_bins):
            mask = bin_idx_per_cell == bi
            cell_idxs = idxs[mask]
            n_total = mask.sum()
            if n_total == 0:
                bin_records.append(None)
                bin_tissue_provisional.append(None)
                continue

            n_epi = is_epi[cell_idxs].sum()
            n_cd8 = is_cd8[cell_idxs].sum()
            epi_frac = n_epi / n_total

            gdf15_total = gdf15[cell_idxs].sum()
            cd8a_total  = cd8a[cell_idxs].sum()
            cd3e_total  = cd3e[cell_idxs].sum()

            bin_id = f"{core_id}|{bi}"
            bin_id_per_cell[cell_idxs] = bin_id

            has_epi = n_epi >= 1
            bin_has_epi[bi] = has_epi

            
            if not is_mixed_core:
                bt = core_label
            else:
                if has_epi:
                    epi_tissues = tissue_arr[cell_idxs[is_epi[cell_idxs]]]
                    epi_tissues = epi_tissues[
                        (epi_tissues != 'nan')
                        & (np.char.upper(epi_tissues.astype(str)) != 'NA')
                    ]
                    if len(epi_tissues) == 0:
                        bt = None  
                        bin_has_epi[bi] = False
                    else:
                        vals, counts_ = np.unique(epi_tissues, return_counts=True)
                        bt = str(vals[np.argmax(counts_)])
                else:
                    bt = None  

            bin_tissue_provisional.append(bt)

            bin_records.append({
                'core':          core_id,
                'bin_id':        bin_id,
                'bin_idx':       int(bi),
                'tissue':        bt,             
                'bin_x':         float(center_x[bi]),
                'bin_y':         float(center_y[bi]),
                'n_total':       int(n_total),
                'n_epi':         int(n_epi),
                'n_nonepi':      int(n_total - n_epi),
                'epi_fraction':  float(epi_frac),
                'n_cd8':         int(n_cd8),
                'gdf15_total':   float(gdf15_total),
                'cd8a_total':    float(cd8a_total),
                'cd3e_total':    float(cd3e_total),
            })

        
        if is_mixed_core:
            n_mixed_cores_seen += 1
            valid_mask = np.array([rec is not None for rec in bin_records])
            valid_idxs = np.flatnonzero(valid_mask)
            n_mixed_bins_total += valid_idxs.size

            epi_bin_idxs = [
                bi for bi in valid_idxs
                if bin_has_epi[bi] and bin_tissue_provisional[bi] is not None
            ]
            non_epi_bin_idxs = [
                bi for bi in valid_idxs
                if not bin_has_epi[bi]
            ]
            n_mixed_bins_epi += len(epi_bin_idxs)

            if len(epi_bin_idxs) == 0:
                
                n_mixed_bins_no_epi_in_core += len(non_epi_bin_idxs)
                for bi in non_epi_bin_idxs:
                    bin_records[bi]['tissue'] = 'Mixed'
            else:
                epi_centers = np.column_stack([
                    [center_x[bi] for bi in epi_bin_idxs],
                    [center_y[bi] for bi in epi_bin_idxs],
                ])
                epi_labels = np.array(
                    [bin_tissue_provisional[bi] for bi in epi_bin_idxs]
                )

                for bi in non_epi_bin_idxs:
                    dx = epi_centers[:, 0] - center_x[bi]
                    dy = epi_centers[:, 1] - center_y[bi]
                    d2 = dx * dx + dy * dy
                    nearest = int(np.argmin(d2))
                    bin_records[bi]['tissue'] = str(epi_labels[nearest])
                    n_mixed_bins_inherited += 1

        
        for rec in bin_records:
            if rec is not None:
                rows.append(rec)

    if n_mixed_cores_seen:
        print(f"  Mixed cores: {n_mixed_cores_seen}")
        print(f"    bins total in Mixed cores: {n_mixed_bins_total}")
        print(f"    epi bins (kept own tissue):  {n_mixed_bins_epi}")
        print(f"    non-epi bins (nearest-epi inherited): {n_mixed_bins_inherited}")
        if n_mixed_bins_no_epi_in_core:
            print(f"    non-epi bins in Mixed cores with NO epi bins "
                  f"(labeled 'Mixed'): {n_mixed_bins_no_epi_in_core}")

    return pd.DataFrame(rows), bin_id_per_cell, bin_geometry


# ─── Bivariate Moran's per core ────────────────────────────────────────────
def run_bivariate_morans_per_core(
    bins_df, bin_geometry,
    x_col='gdf15_total', y_col='cd8a_total',
    n_permutations=N_PERMUTATIONS, seed=LISA_SEED,
):
    df = bins_df.copy()
    df['bv_I'] = np.nan
    df['bv_z'] = np.nan
    df['bv_p'] = np.nan
    df['bv_q'] = pd.NA
    df['bv_quadrant'] = 'NOT_RUN'

    rng = np.random.default_rng(seed)
    skipped = []

    for core_id, grp in df.groupby('core'):
        if len(grp) < 5:
            skipped.append((core_id, f'<5 bins ({len(grp)})'))
            continue
        coords = grp[['bin_x', 'bin_y']].values
        x_vals = grp[x_col].values.astype(float)
        y_vals = grp[y_col].values.astype(float)
        if np.var(x_vals) == 0:
            skipped.append((core_id, f'no variance in {x_col}'))
            continue
        if np.var(y_vals) == 0:
            skipped.append((core_id, f'no variance in {y_col} (all = {y_vals[0]:.1f})'))
            continue

        threshold = 1.5 * bin_geometry[core_id]['bin_size_um']
        try:
            w = DistanceBand.from_array(
                coords, threshold=threshold,
                binary=True, silence_warnings=True,
            )
        except Exception as e:
            skipped.append((core_id, f'weights failed: {e}'))
            continue
        if w.mean_neighbors < 1:
            skipped.append(
                (core_id, f'mean_neighbors < 1 (threshold={threshold:.1f})')
            )
            continue

        np.random.seed(int(rng.integers(0, 2**31 - 1)))
        bv = Moran_Local_BV(x_vals, y_vals, w, permutations=n_permutations)

        df.loc[grp.index, 'bv_I'] = bv.Is
        df.loc[grp.index, 'bv_z'] = bv.z_sim
        df.loc[grp.index, 'bv_p'] = bv.p_sim
        df.loc[grp.index, 'bv_q'] = bv.q
        df.loc[grp.index, 'bv_quadrant'] = [
            BV_QUADRANT[q] if p < 0.05 else 'NS'
            for q, p in zip(bv.q, bv.p_sim)
        ]

    if skipped:
        from collections import Counter
        print(f"  Skipped {len(skipped)} cores (Moran's didn't run):")
        reasons = Counter(reason.split(' (')[0] for _, reason in skipped)
        for reason, n in reasons.most_common():
            print(f"    {n}× {reason}")

    return df


# ─── Reversed driver: CD8A/CD3E focal × GDF15 lag ──────────────────────────
def run_reversed_bivariate_lisa_all_cells(
    adata, run_name,
    bin_size_um=BIN_SIZE_UM,
    n_permutations=N_PERMUTATIONS, seed=LISA_SEED,
):
    print(f"Square-bin aggregation — ALL CELLS (bin_size_um={bin_size_um}, "
          f"keeping any bin with >=1 cell)...")
    bins_df, bin_id_per_cell, bin_geometry = aggregate_to_squarebins_all_cells(
        adata, bin_size_um=bin_size_um,
    )
    print(f"  {len(bins_df)} bins across {bins_df['core'].nunique()} cores")
    if len(bins_df):
        print(f"  median cells/bin = {bins_df['n_total'].median():.0f}, "
              f"median epi_fraction = {bins_df['epi_fraction'].median():.2f}")
        print(f"  median gdf15_total = {bins_df['gdf15_total'].median():.1f}, "
              f"median cd8a_total = {bins_df['cd8a_total'].median():.1f}, "
              f"median cd3e_total = {bins_df['cd3e_total'].median():.1f}")
        print(f"  bins per tissue: "
              f"{bins_df['tissue'].value_counts().to_dict()}")

    print("\nReversed Moran's: CD8A (focal) × GDF15 (lag)...")
    bins_cd8a_rev = run_bivariate_morans_per_core(
        bins_df, bin_geometry,
        x_col='cd8a_total', y_col='gdf15_total',
        n_permutations=n_permutations, seed=seed,
    )
    print(f"  Quadrants: {bins_cd8a_rev['bv_quadrant'].value_counts().to_dict()}")

    print("\nReversed Moran's: CD3E (focal) × GDF15 (lag)...")
    bins_cd3e_rev = run_bivariate_morans_per_core(
        bins_df, bin_geometry,
        x_col='cd3e_total', y_col='gdf15_total',
        n_permutations=n_permutations, seed=seed,
    )
    print(f"  Quadrants: {bins_cd3e_rev['bv_quadrant'].value_counts().to_dict()}")

    suf = f"__{run_name}__rev_allcells"
    adata.obs[f'bv_lisa_bin_id{suf}'] = bin_id_per_cell
    cd8a_map = bins_cd8a_rev.set_index('bin_id')['bv_quadrant'].to_dict()
    cd3e_map = bins_cd3e_rev.set_index('bin_id')['bv_quadrant'].to_dict()
    adata.obs[f'bv_lisa_cd8a_gdf15{suf}'] = (
        adata.obs[f'bv_lisa_bin_id{suf}'].map(cd8a_map).fillna('')
    )
    adata.obs[f'bv_lisa_cd3e_gdf15{suf}'] = (
        adata.obs[f'bv_lisa_bin_id{suf}'].map(cd3e_map).fillna('')
    )

    return bins_cd8a_rev, bins_cd3e_rev, bin_geometry


# ─── Square rendering helper ───────────────────────────────────────────────
def _draw_square_collection(ax, core_bins, bin_geometry, core_id):
    if core_bins.empty:
        return
    size = bin_geometry[core_id]['bin_size_um'] if core_id in bin_geometry else BIN_SIZE_UM

    patches = []
    colors = []
    for _, row in core_bins.iterrows():
        q = row['bv_quadrant']
        c = QUADRANT_COLORS.get(q, '#FFFFFF')
        if c == 'none':
            continue
        patches.append(Rectangle(
            (row['bin_x'] - size / 2, row['bin_y'] - size / 2),
            width=size, height=size,
        ))
        colors.append(c)

    if patches:
        ax.add_collection(PatchCollection(
            patches, facecolors=colors,
            edgecolors='white', linewidths=0.4,
        ))


# ─── Ranking (sorted by LH by default) ─────────────────────────────────────
def rank_cores_by_tcell_isolation(
    bins_df, adata,
    min_high_tcell_bins=3,
    sort_by='n_LH',
):
    """
    Rank cores from the reversed run.
      n_HL = High CD8A focal, Low GDF15 nearby  (T-cell hotspot away from GDF15)
      n_LH = Low CD8A focal, High GDF15 nearby  (GDF15 hotspot with no T-cells in it)
    """
    obs = adata.obs

    if CORE_TISSUE_COL not in obs.columns:
        raise ValueError(
            f"adata.obs is missing {CORE_TISSUE_COL!r}. "
            f"Build it before ranking."
        )
    core_tissue = (
        obs[[CORE_COL, CORE_TISSUE_COL]]
        .drop_duplicates()
        .set_index(CORE_COL)[CORE_TISSUE_COL]
        .astype(str)
    )
    patient_map = (
        obs[[CORE_COL, PATIENT_COL]].drop_duplicates()
           .set_index(CORE_COL)[PATIENT_COL]
    )

    rows = []
    for core_id, grp in bins_df.groupby('core'):
        n_HH = (grp['bv_quadrant'] == 'HH').sum()
        n_HL = (grp['bv_quadrant'] == 'HL').sum()
        n_LH = (grp['bv_quadrant'] == 'LH').sum()
        n_LL = (grp['bv_quadrant'] == 'LL').sum()
        n_NS = (grp['bv_quadrant'] == 'NS').sum()
        n_high_tcell = n_HH + n_HL
        rows.append({
            'core': core_id,
            'tissue': core_tissue.get(core_id, '?'),  
            'patient': patient_map.get(core_id, '?'),
            'n_bins': len(grp),
            'n_HH': n_HH, 'n_HL': n_HL,
            'n_LH': n_LH, 'n_LL': n_LL, 'n_NS': n_NS,
            'n_high_tcell': n_high_tcell,
            'isolated_fraction': (n_HL / n_high_tcell) if n_high_tcell > 0 else np.nan,
        })

    return (
        pd.DataFrame(rows)
        .query(f'n_high_tcell >= {min_high_tcell_bins}')
        .sort_values(sort_by, ascending=False)
        .reset_index(drop=True)
    )


# ─── Top-N grid ────────────────────────────────────────────────────────────
def plot_squarebins_by_quadrant(
    bins_df, ranking_df, adata, bin_geometry,
    n_top=12, ncols=4,
    title='Bivariate LISA square bins',
    quadrant_legend_labels=None,
):
    if quadrant_legend_labels is None:
        quadrant_legend_labels = {
            'HL': 'HL — High GDF15, Low T (EXCLUSION)',
            'HH': 'HH — High GDF15, High T',
            'LH': 'LH — Low GDF15, High T',
            'LL': 'LL — Low GDF15, Low T',
            'NS': 'NS — Not significant',
        }

    cores = ranking_df.head(n_top)['core'].tolist()
    if not cores:
        print("Nothing to plot")
        return None

    n_rows = int(np.ceil(len(cores) / ncols))
    fig, axes = plt.subplots(
        n_rows, ncols,
        figsize=(5 * ncols, 5 * n_rows),
        squeeze=False,
    )
    axes_flat = axes.flatten()

    for idx, core_id in enumerate(cores):
        ax = axes_flat[idx]
        info = ranking_df[ranking_df['core'] == core_id].iloc[0]
        core_bins = bins_df[bins_df['core'] == core_id]

        if core_bins.empty:
            ax.set_title(f"{core_id}\n(no bins passed filter)")
            ax.axis('off')
            continue

        _draw_square_collection(ax, core_bins, bin_geometry, core_id)

        ax.set_aspect('equal')
        ax.autoscale_view()
        ax.set_xticks([])
        ax.set_yticks([])
        ylim = ax.get_ylim()
        ax.set_ylim(max(ylim), min(ylim))
        ax.set_title(
            f"{core_id} [{info['tissue']}] (pt {info['patient']})\n"
            f"LH={info['n_LH']}, HH={info['n_HH']}, total bins={info['n_bins']}",
            fontsize=10, fontweight='bold',
        )

    for i in range(len(cores), len(axes_flat)):
        axes_flat[i].set_visible(False)

    legend_elements = [
        Patch(facecolor=QUADRANT_COLORS[q], label=quadrant_legend_labels[q])
        for q in ['HL', 'HH', 'LH', 'LL', 'NS']
    ]
    fig.legend(handles=legend_elements, loc='upper center',
               ncol=5, fontsize=10, bbox_to_anchor=(0.5, 0.02), frameon=False)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0.03, 1, 0.99])
    return fig


# ─── Stacked-bar tissue summary ────────────────────────────────────────────
def plot_quadrant_proportions_by_tissue(
    bins_df, adata,
    title='Quadrant proportions by tissue type',
    quadrant_order=['HL', 'HH', 'LH', 'LL'],
    tissue_order=None,
    normalize=True,
    figsize=(10, 6),
    exclude_not_run=True,
    exclude_ns=True,
    exclude_mixed=True,
    exclude_unknown=True,
    quadrant_legend_labels=None,
):
    if quadrant_legend_labels is None:
        quadrant_legend_labels = {
            'HL': 'HL — High GDF15, Low T (EXCLUSION)',
            'HH': 'HH — High GDF15, High T (co-localized)',
            'LH': 'LH — Low GDF15, High T',
            'LL': 'LL — Low GDF15, Low T',
            'NS': 'NS — Not significant',
        }

    if 'tissue' not in bins_df.columns:
        raise ValueError("bins_df has no 'tissue' column.")

    df = bins_df.copy()

    if exclude_not_run:
        n_before = len(df)
        df = df[df['bv_quadrant'] != 'NOT_RUN']
        print(f"Dropped {n_before - len(df)} NOT_RUN bins ({len(df)} remaining)")
    if exclude_ns:
        n_before = len(df)
        df = df[df['bv_quadrant'] != 'NS']
        print(f"Dropped {n_before - len(df)} NS bins ({len(df)} significant remaining)")
    if exclude_unknown:
        n_before = len(df)
        df = df[df['tissue'] != 'Unknown']
        print(f"Dropped {n_before - len(df)} Unknown-tissue bins")
    if exclude_mixed:

        n_before = len(df)
        df = df[df['tissue'] != 'Mixed']
        print(f"Dropped {n_before - len(df)} residual Mixed-tissue bins "
              f"(Mixed cores w/ no epi bins)")

    if df.empty:
        print("Nothing to plot")
        return None, None

    counts = (
        df.groupby(['tissue', 'bv_quadrant'], observed=True)
          .size().unstack(fill_value=0)
    )
    for q in quadrant_order:
        if q not in counts.columns:
            counts[q] = 0
    counts = counts[quadrant_order]

    if tissue_order is None:
        tissue_order = ['Dist_N', 'Adj_N', 'AD', 'CA']
    counts = counts.reindex([t for t in tissue_order if t in counts.index])

    bin_counts_per_tissue = df['tissue'].value_counts()
    print("Bins per tissue:")
    print(bin_counts_per_tissue.reindex(
        [t for t in tissue_order if t in bin_counts_per_tissue.index]
    ))

    totals = counts.sum(axis=1)
    if normalize:
        plot_data = counts.div(totals.replace(0, np.nan), axis=0).fillna(0)
        ylabel = 'Proportion of significant bins' if exclude_ns else 'Proportion of bins'
    else:
        plot_data = counts
        ylabel = 'Number of significant bins' if exclude_ns else 'Number of bins'

    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(plot_data))
    x = np.arange(len(plot_data))

    for q in quadrant_order:
        vals = plot_data[q].values
        ax.bar(
            x, vals, bottom=bottom,
            color=QUADRANT_COLORS[q],
            edgecolor='white', linewidth=0.6,
            label=quadrant_legend_labels[q],
        )
        if normalize:
            for xi, v in enumerate(vals):
                if v >= 0.04:
                    ax.text(
                        xi, bottom[xi] + v / 2,
                        f'{v*100:.0f}%',
                        ha='center', va='center',
                        fontsize=9,
                        color='white' if q in ('HL', 'HH', 'LH') else 'black',
                        fontweight='bold',
                    )
        bottom += vals

    unit = 'sig bins' if exclude_ns else 'bins'
    n_cores_label = []
    for t in plot_data.index:
        n_c = df.loc[df['tissue'] == t, 'core'].nunique()
        n_cores_label.append(f'{t}\nn={int(totals[t])} {unit}\n({n_c} cores)')

    ax.set_xticks(x)
    ax.set_xticklabels(n_cores_label, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    if normalize:
        ax.set_ylim(0, 1)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return fig, plot_data


# ─── Two-panel per-core plot (ranked by LH) ────────────────────────────────
def plot_top_isolated_cores_two_panel(
    bins_cd8a_rev, adata, bin_geometry,
    n_top=8,
    min_high_tcell_bins=3,
    sort_by='n_LH',
    epi_pos_color='#0c8006',
    epi_neg_color='lightgray',
    cd8_color='#FF00BF',
    cd3e_color='#00C3FF',
    cd8_size=20, cd3e_size=20, epi_size=10,
    figsize_per_panel=(6, 6),
    gdf15_gene=GDF15_GENE,
):
    ranking = rank_cores_by_tcell_isolation(
        bins_cd8a_rev, adata,
        min_high_tcell_bins=min_high_tcell_bins,
        sort_by=sort_by,
    )
    cores = ranking.head(n_top)['core'].tolist()
    if not cores:
        print("No cores to plot")
        return []

    obs = adata.obs
    coords_all = adata.obsm['spatial']

    if gdf15_gene not in adata.var_names:
        raise ValueError(f"{gdf15_gene} not in adata.var_names")
    if 'counts' in adata.layers:
        gdf15_full = adata[:, gdf15_gene].layers['counts']
    else:
        gdf15_full = adata[:, gdf15_gene].X
    gdf15_full = (
        gdf15_full.toarray().flatten()
        if hasattr(gdf15_full, 'toarray')
        else np.asarray(gdf15_full).flatten()
    )

    figures = []

    for core_id in cores:
        info = ranking[ranking['core'] == core_id].iloc[0]
        tissue = info['tissue']
        patient = info['patient']

        fig, axes = plt.subplots(
            1, 2,
            figsize=(figsize_per_panel[0] * 2, figsize_per_panel[1]),
        )
        ax1, ax2 = axes

        cd8a_core_bins = bins_cd8a_rev[bins_cd8a_rev['core'] == core_id]
        _draw_square_collection(ax1, cd8a_core_bins, bin_geometry, core_id)
        ax1.set_title("CD8A (focal) × GDF15 (lag)", fontsize=10, fontweight='bold')

        sel = obs[CORE_COL].values == core_id
        cc = coords_all[sel]
        ct_arr_core = obs[CELLTYPE_COL].values[sel]
        gdf15_core = gdf15_full[sel]

        is_epi = ct_arr_core == EPITHELIAL_LABEL
        epi_coords = cc[is_epi]
        epi_gdf15 = gdf15_core[is_epi]
        neg_mask = epi_gdf15 == 0
        pos_mask = epi_gdf15 > 0

        scatter_handles = []
        if neg_mask.any():
            h = ax2.scatter(epi_coords[neg_mask, 0], epi_coords[neg_mask, 1],
                            c=epi_neg_color, s=epi_size, alpha=0.4, zorder=2,
                            label=f'GDF15− epi (n={neg_mask.sum()})')
            scatter_handles.append(h)
        if pos_mask.any():
            h = ax2.scatter(epi_coords[pos_mask, 0], epi_coords[pos_mask, 1],
                            c=epi_pos_color, s=epi_size + 4, alpha=0.85, zorder=4,
                            edgecolors='white', linewidths=0.2,
                            label=f'GDF15+ epi (n={pos_mask.sum()})')
            scatter_handles.append(h)

        tcell_coords = cc[ct_arr_core == TCELL_LABEL]
        cd8_coords   = cc[ct_arr_core == CD8_LABEL]
        if len(tcell_coords):
            h = ax2.scatter(tcell_coords[:, 0], tcell_coords[:, 1],
                            c=cd3e_color, s=cd3e_size, alpha=0.9, zorder=5, linewidths=0,
                            label=f'T cells non-CD8 (n={len(tcell_coords)})')
            scatter_handles.append(h)
        if len(cd8_coords):
            h = ax2.scatter(cd8_coords[:, 0], cd8_coords[:, 1],
                            c=cd8_color, s=cd8_size, alpha=0.9, zorder=6, linewidths=0,
                            label=f'CD8 T cells (n={len(cd8_coords)})')
            scatter_handles.append(h)

        ax2.set_title('Cell positions', fontsize=10, fontweight='bold')

        x_min, x_max = cc[:, 0].min() - 50, cc[:, 0].max() + 50
        y_min, y_max = cc[:, 1].min() - 50, cc[:, 1].max() + 50
        for ax in (ax1, ax2):
            ax.set_aspect('equal')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)
            for spine in ax.spines.values():
                spine.set_visible(False)

        fig.subplots_adjust(
            wspace=0.02, left=0.18, right=0.82, top=0.92, bottom=0.22,
        )

        quadrant_handles = [
            Patch(facecolor=QUADRANT_COLORS['HL'],
                  label='HL — High CD8A, Low GDF15 nearby'),
            Patch(facecolor=QUADRANT_COLORS['HH'],
                  label='HH — High CD8A, High GDF15 nearby (co-localized)'),
            Patch(facecolor=QUADRANT_COLORS['LH'],
                  label='LH — Low CD8A, High GDF15 nearby (T-cells absent)'),
            Patch(facecolor=QUADRANT_COLORS['LL'],
                  label='LL — Low CD8A, Low GDF15 nearby'),
            Patch(facecolor=QUADRANT_COLORS['NS'],
                  label='NS — Not significant'),
        ]

        leg1 = fig.legend(
            handles=quadrant_handles,
            loc='upper center', bbox_to_anchor=(0.34, 0.18),
            ncol=1, fontsize=9, frameon=False,
        )
        fig.add_artist(leg1)

        fig.legend(
            handles=scatter_handles,
            loc='upper center', bbox_to_anchor=(0.66, 0.18),
            ncol=1, fontsize=9, frameon=False,
        )

        fig.suptitle(
            f'{core_id} [{tissue}] — LH={info["n_LH"]}',
            fontsize=13, fontweight='bold', y=0.98,
        )

        figures.append((core_id, fig))

    return figures


# ─── Reversed-direction legend labels ──────────────────────────────────────
REV_LEGEND_CD8A = {
    'HL': 'HL — High CD8A, Low GDF15 nearby',
    'HH': 'HH — High CD8A, High GDF15 nearby (co-localized)',
    'LH': 'LH — Low CD8A, High GDF15 nearby (T-cells absent)',
    'LL': 'LL — Low CD8A, Low GDF15 nearby',
    'NS': 'NS — Not significant',
}
REV_LEGEND_CD3E = {
    'HL': 'HL — High CD3E, Low GDF15 nearby',
    'HH': 'HH — High CD3E, High GDF15 nearby (co-localized)',
    'LH': 'LH — Low CD3E, High GDF15 nearby (T-cells absent)',
    'LL': 'LL — Low CD3E, Low GDF15 nearby',
    'NS': 'NS — Not significant',
}


# ─── Run reversed pipeline ─────────────────────────────────────────────────
bins_cd8a_rev, bins_cd3e_rev, bin_geometry_rev = run_reversed_bivariate_lisa_all_cells(
    adata, run_name,
    bin_size_um=BIN_SIZE_UM,
)




In [ ]:
# ─── Stacked-bar tissue summary ────────────────────────────────────────────
def plot_quadrant_proportions_by_tissue(
    bins_df, adata,
    title='Quadrant proportions by tissue type',
    quadrant_order=['HL', 'HH', 'LH', 'LL'],
    tissue_order=None,
    normalize=True,
    figsize=(10, 6),
    exclude_not_run=True,
    exclude_ns=True,
    exclude_mixed=True,
    exclude_unknown=True,
    quadrant_legend_labels=None,
):
    if quadrant_legend_labels is None:
        quadrant_legend_labels = {
            'HL': 'HL — High GDF15, Low T (EXCLUSION)',
            'HH': 'HH — High GDF15, High T (co-localized)',
            'LH': 'LH — Low GDF15, High T',
            'LL': 'LL — Low GDF15, Low T',
            'NS': 'NS — Not significant',
        }

    if 'tissue' not in bins_df.columns:
        raise ValueError("bins_df has no 'tissue' column.")

    df = bins_df.copy()

    if exclude_not_run:
        n_before = len(df)
        df = df[df['bv_quadrant'] != 'NOT_RUN']
        print(f"Dropped {n_before - len(df)} NOT_RUN bins ({len(df)} remaining)")
    if exclude_ns:
        n_before = len(df)
        df = df[df['bv_quadrant'] != 'NS']
        print(f"Dropped {n_before - len(df)} NS bins ({len(df)} significant remaining)")
    if exclude_unknown:
        n_before = len(df)
        df = df[df['tissue'] != 'Unknown']
        print(f"Dropped {n_before - len(df)} Unknown-tissue bins")
    if exclude_mixed:
        # With nearest-epi inheritance, 'Mixed' only remains for bins in
        # Mixed cores that had no epi bins anywhere — drop them.
        n_before = len(df)
        df = df[df['tissue'] != 'Mixed']
        print(f"Dropped {n_before - len(df)} residual Mixed-tissue bins "
              f"(Mixed cores w/ no epi bins)")

    if df.empty:
        print("Nothing to plot")
        return None, None

    counts = (
        df.groupby(['tissue', 'bv_quadrant'], observed=True)
          .size().unstack(fill_value=0)
    )
    for q in quadrant_order:
        if q not in counts.columns:
            counts[q] = 0
    counts = counts[quadrant_order]

    if tissue_order is None:
        tissue_order = ['Dist_N', 'Adj_N', 'AD', 'CA']
    counts = counts.reindex([t for t in tissue_order if t in counts.index])

    bin_counts_per_tissue = df['tissue'].value_counts()
    print("Bins per tissue:")
    print(bin_counts_per_tissue.reindex(
        [t for t in tissue_order if t in bin_counts_per_tissue.index]
    ))

    totals = counts.sum(axis=1)
    if normalize:
        plot_data = counts.div(totals.replace(0, np.nan), axis=0).fillna(0)
        ylabel = 'Proportion of significant bins' if exclude_ns else 'Proportion of bins'
    else:
        plot_data = counts
        ylabel = 'Number of significant bins' if exclude_ns else 'Number of bins'

    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(plot_data))
    x = np.arange(len(plot_data))

    for q in quadrant_order:
        vals = plot_data[q].values
        ax.bar(
            x, vals, bottom=bottom,
            color=QUADRANT_COLORS[q],
            edgecolor='white', linewidth=0.6,
            label=quadrant_legend_labels[q],
        )
        bottom += vals

    unit = 'sig bins' if exclude_ns else 'bins'
    n_cores_label = []
    for t in plot_data.index:
        n_c = df.loc[df['tissue'] == t, 'core'].nunique()
        n_cores_label.append(f'{t}\nn={int(totals[t])}\n({n_c} cores)')

    ax.set_xticks(x)
    ax.set_xticklabels(n_cores_label, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold')
    if normalize:
        ax.set_ylim(0, 1)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return fig, plot_data


REV_LEGEND_CD8A = {
    'HL': 'HL — High CD8A, Low GDF15 nearby',
    'HH': 'HH — High CD8A, High GDF15 nearby',
    'LH': 'LH — Low CD8A, High GDF15 nearby',
    'LL': 'LL — Low CD8A, Low GDF15 nearby',
    'NS': 'NS — Not significant',
}


# fig_props_cd8a_rev, props_cd8a_rev = plot_quadrant_proportions_by_tissue(
#     bins_cd8a_rev, adata,
#     title='CD8A × GDF15 Local Bivariate Morans I',
#     quadrant_legend_labels=REV_LEGEND_CD8A,
# )
# plt.show()


In [ ]:
# ─── Stacked bar with Dist_N merged into N ─────────────────────────────
bins_cd8a_rev_merged = bins_cd8a_rev.copy()
bins_cd8a_rev_merged['tissue'] = bins_cd8a_rev_merged['tissue'].replace(
    {'Dist_N': 'Adj_N'}
)

def add_significance_brackets(
    ax,
    pairs,
    group_positions,
    y_start=1.02,
    bracket_h=0.02,
    spacing=0.08,
    fontsize=11,
    color='black',
    quadrant_colors=None,
):
    """Draw significance brackets above stacked bars.

    pairs: list of (g1, g2, segments) tuples where segments is a list of
        (quadrant_label, asterisks) tuples — e.g.
        [('Adj_N', 'AD', [('HH', '**'), ('LH', '**')]),
         ('AD', 'CA', [('HH', '*')])]
    group_positions: dict mapping tissue name → x position
    quadrant_colors: dict mapping quadrant label → color (for bold colored labels)
    """
    from matplotlib.transforms import blended_transform_factory
    trans = ax.get_xaxis_transform()  # x in data, y in axis fraction

    for i, (g1, g2, segments) in enumerate(pairs):
        x1, x2 = group_positions[g1], group_positions[g2]
        y = y_start + i * spacing
        x_mid = (x1 + x2) / 2

        # Bracket
        ax.plot(
            [x1, x1, x2, x2],
            [y, y + bracket_h, y + bracket_h, y],
            transform=trans, lw=1.2, color=color, clip_on=False,
        )


        renderer = ax.figure.canvas.get_renderer()
        full_text = '   '.join(f'{q}{a}' for q, a in segments)
        tmp = ax.text(
            x_mid, y + bracket_h + 0.005, full_text,
            transform=trans, ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold', alpha=0,
            clip_on=False,
        )
        bbox = tmp.get_window_extent(renderer=renderer)
        inv = trans.inverted()
        x_left_data, _ = inv.transform((bbox.x0, bbox.y0))
        x_right_data, _ = inv.transform((bbox.x1, bbox.y0))
        tmp.remove()


        cur_x = x_left_data
        for j, (quad, ast) in enumerate(segments):
            if j > 0:

                spacer = ax.text(
                    cur_x, y + bracket_h + 0.005, '   ',
                    transform=trans, ha='left', va='bottom',
                    fontsize=fontsize, fontweight='bold', alpha=0,
                    clip_on=False,
                )
                sb = spacer.get_window_extent(renderer=renderer)
                cur_x = inv.transform((sb.x1, sb.y0))[0]
                spacer.remove()

            piece = f'{quad}{ast}'
            piece_color = (
                quadrant_colors.get(quad, color)
                if quadrant_colors else color
            )
            t = ax.text(
                cur_x, y + bracket_h + 0.005, piece,
                transform=trans, ha='left', va='bottom',
                fontsize=fontsize, fontweight='bold',
                color=piece_color, clip_on=False,
            )
            tb = t.get_window_extent(renderer=renderer)
            cur_x = inv.transform((tb.x1, tb.y0))[0]


# Plot
fig_props_cd8a_rev_merged, props_cd8a_rev_merged = plot_quadrant_proportions_by_tissue(
    bins_cd8a_rev_merged, adata,
    title='CD8A × GDF15 Local Bivariate Morans I',
    tissue_order=['Adj_N', 'AD', 'CA'],
    quadrant_legend_labels=REV_LEGEND_CD8A,
)

ax = fig_props_cd8a_rev_merged.axes[0]


ax.set_title(
    'CD8A × GDF15 Local Bivariate Morans I',
    fontsize=14, fontweight='bold', pad=70,
)

fig_props_cd8a_rev_merged.suptitle('')

add_significance_brackets(
    ax,
    pairs=[
        ('Adj_N',  'AD', [('HH', '**'), ('LH', '**')]),
        ('AD', 'CA', [('HH', '**')]),
    ],
    group_positions={'Adj_N': 0, 'AD': 1, 'CA': 2},
    quadrant_colors=QUADRANT_COLORS,   # uses your existing palette
)

fig_props_cd8a_rev_merged.savefig(
    f'{output_dir}/cd8a_gdf15_quadrant_proportions.pdf',
    dpi=300, bbox_inches='tight',
)
plt.show()

In [ ]:
def _resolve_cores(bins_cd8a_rev, adata, n_top, min_high_tcell_bins,
                   sort_by, core_ids):
    """Shared core-resolution logic for both functions."""
    if core_ids is not None:
        ranking = rank_cores_by_tcell_isolation(
            bins_cd8a_rev, adata,
            min_high_tcell_bins=0,
            sort_by=sort_by,
        )
        cores = list(core_ids)
        available = set(ranking['core'])
        missing = [c for c in cores if c not in available]
        if missing:
            print(f"Warning: {len(missing)} core(s) not found and will be skipped: {missing}")
            cores = [c for c in cores if c in available]
    else:
        ranking = rank_cores_by_tcell_isolation(
            bins_cd8a_rev, adata,
            min_high_tcell_bins=min_high_tcell_bins,
            sort_by=sort_by,
        )
        cores = ranking.head(n_top)['core'].tolist()
    return cores, ranking


def _get_gdf15_full(adata, gdf15_gene):
    if gdf15_gene not in adata.var_names:
        raise ValueError(f"{gdf15_gene} not in adata.var_names")
    if 'normalized' not in adata.layers:
        raise ValueError("adata.layers['normalized'] not found")
    gdf15_full = adata[:, gdf15_gene].layers['normalized']
    return (
        gdf15_full.toarray().flatten()
        if hasattr(gdf15_full, 'toarray')
        else np.asarray(gdf15_full).flatten()
    )


def _compute_shared_extent(cores, obs, coords_all, pad=50):
    """Largest spatial span (max of x and y range) across all cores, plus pad.
    Used so every core panel renders at the same visible size."""
    max_span = 0.0
    for core_id in cores:
        sel = obs[CORE_COL].values == core_id
        if not sel.any():
            continue
        cc = coords_all[sel]
        x_span = cc[:, 0].max() - cc[:, 0].min()
        y_span = cc[:, 1].max() - cc[:, 1].min()
        max_span = max(max_span, x_span, y_span)
    return max_span + 2 * pad



TISSUE_ORDER = ['Adj_N', 'AD', 'CA']


TISSUE_DISPLAY = {'Dist_N': 'Adj_N'}


def _tissue_sort_key(tissue):
    """Return a sort key that puts N first, then TA, then CA, then everything else."""
    t = str(tissue)
    for i, prefix in enumerate(sorted(TISSUE_ORDER, key=len, reverse=True)):
        if t.startswith(prefix):
            return (TISSUE_ORDER.index(prefix), t)
    return (len(TISSUE_ORDER), t)


def _draw_core_row(axes_row, core_id, ranking, bins_cd8a_rev, bin_geometry,
                   obs, coords_all, gdf15_full,
                   cd8_color, cd3e_color,
                   cd8_size, cd3e_size, epi_size,
                   gdf15_gene, gdf15_cmap, show_tcells,
                   add_colorbar=True, shared_extent=None,
                   show_titles=True, col_title_fontsize=14):
    """Draw a single core's three panels (LISA bins, GDF15+T cells, blank H&E)
    onto a row of three axes. Returns the display label string for the core.

    shared_extent : float or None
        If given, every panel is drawn with the same x/y span (this many units
        wide and tall), centered on the core's centroid.
    show_titles : bool
        If True, set per-panel column titles.
    col_title_fontsize : int
        Font size for the column titles.
    """
    import matplotlib.colors as mcolors

    ax_he, ax1, ax2 = axes_row

    info = ranking[ranking['core'] == core_id].iloc[0]
    tissue = info['tissue']
    patient = info['patient']

    # Panel 1 (leftmost): blank H&E
    if show_titles:
        ax_he.set_title('H&E', fontsize=col_title_fontsize, fontweight='bold')

    cd8a_core_bins = bins_cd8a_rev[bins_cd8a_rev['core'] == core_id]
    _draw_square_collection(ax1, cd8a_core_bins, bin_geometry, core_id)
    for art in ax1.collections + ax1.patches:
        art.set_rasterized(True)
    if show_titles:
        ax1.set_title("CD8A × GDF15",
                      fontsize=col_title_fontsize, fontweight='bold')

    sel = obs[CORE_COL].values == core_id
    cc = coords_all[sel]
    ct_arr_core = obs[CELLTYPE_COL].values[sel]
    gdf15_core = gdf15_full[sel]

    is_epi = ct_arr_core == EPITHELIAL_LABEL
    epi_coords = cc[is_epi]
    epi_gdf15 = gdf15_core[is_epi]

    if len(epi_coords):
        order = np.argsort(epi_gdf15)
        vmax = (
            np.percentile(epi_gdf15[epi_gdf15 > 0], 99)
            if (epi_gdf15 > 0).any() else 1.0
        )
        sc = ax2.scatter(
            epi_coords[order, 0], epi_coords[order, 1],
            c=epi_gdf15[order],
            cmap=gdf15_cmap,
            norm=mcolors.Normalize(vmin=0, vmax=vmax),
            s=epi_size + 2, alpha=0.9, zorder=3,
            edgecolors='none',
            rasterized=True,
        )
        if add_colorbar:
            cbar = ax2.figure.colorbar(sc, ax=ax2, fraction=0.03, pad=0.02)
            cbar.set_label(f'{gdf15_gene} (normalized)', fontsize=9)
            cbar.ax.tick_params(labelsize=8)

    tcell_coords = cc[ct_arr_core == TCELL_LABEL]
    cd8_coords   = cc[ct_arr_core == CD8_LABEL]
    if show_tcells and len(tcell_coords):
        ax2.scatter(tcell_coords[:, 0], tcell_coords[:, 1],
                    c=cd3e_color, s=cd3e_size, alpha=0.9, zorder=5,
                    linewidths=0, rasterized=True)
    if len(cd8_coords):
        ax2.scatter(cd8_coords[:, 0], cd8_coords[:, 1],
                    c=cd8_color, s=cd8_size, alpha=0.95, zorder=6,
                    edgecolors='white', linewidths=0.4, rasterized=True)

    if show_titles:
        ax2.set_title(f'{gdf15_gene} expression + T Cells',
                      fontsize=col_title_fontsize, fontweight='bold')

    if shared_extent is not None:
        cx = (cc[:, 0].min() + cc[:, 0].max()) / 2.0
        cy = (cc[:, 1].min() + cc[:, 1].max()) / 2.0
        half = shared_extent / 2.0
        x_min, x_max = cx - half, cx + half
        y_min, y_max = cy - half, cy + half
    else:
        x_min, x_max = cc[:, 0].min() - 50, cc[:, 0].max() + 50
        y_min, y_max = cc[:, 1].min() - 50, cc[:, 1].max() + 50
    for ax in (ax_he, ax1, ax2):
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_max, y_min)
        for spine in ax.spines.values():
            spine.set_visible(False)

    return TISSUE_DISPLAY.get(tissue, tissue)


def _make_legend_figure(cd8_color, cd3e_color):
    quadrant_handles = [
        Patch(facecolor=QUADRANT_COLORS['HL'],
              label='HL — High CD8A, Low GDF15 nearby'),
        Patch(facecolor=QUADRANT_COLORS['HH'],
              label='HH — High CD8A, High GDF15 nearby'),
        Patch(facecolor=QUADRANT_COLORS['LH'],
              label='LH — Low CD8A, High GDF15 nearby'),
        Patch(facecolor=QUADRANT_COLORS['LL'],
              label='LL — Low CD8A, Low GDF15 nearby'),
        Patch(facecolor=QUADRANT_COLORS['NS'],
              label='NS — Not significant'),
    ]
    scatter_legend_handles = [
        Patch(facecolor=cd3e_color, label='T cells (non-CD8)'),
        Patch(facecolor=cd8_color, label='CD8 T cells'),
    ]

    fig_legend, ax_legend = plt.subplots(figsize=(10, 3))
    ax_legend.axis('off')
    leg1 = ax_legend.legend(
        handles=quadrant_handles,
        loc='center left', bbox_to_anchor=(0.0, 0.5),
        ncol=1, fontsize=10, frameon=False,
        title='LISA quadrants', title_fontsize=11,
    )
    ax_legend.add_artist(leg1)
    ax_legend.legend(
        handles=scatter_legend_handles,
        loc='center left', bbox_to_anchor=(0.55, 0.5),
        ncol=1, fontsize=10, frameon=False,
        title='Cell types', title_fontsize=11,
    )
    return fig_legend


def plot_top_isolated_cores_two_panel(
    bins_cd8a_rev, adata, bin_geometry,
    n_top=8,
    min_high_tcell_bins=3,
    sort_by='n_LH',
    core_ids=None,
    cd8_color="#FF0066",
    cd3e_color='#00C3FF',
    cd8_size=50, cd3e_size=20, epi_size=10,
    figsize_per_panel=(6, 6),
    gdf15_gene=GDF15_GENE,
    gdf15_cmap='Greens',
    show_tcells=True,
    col_title_fontsize=14,
    tissue_filter=None,
):
    """One figure per core, three panels: LISA bins, GDF15+T cells, blank H&E.

    Parameters
    ----------
    tissue_filter : tuple of str or None
        If given, only cores whose tissue is in this set are plotted.
        E.g. ('Adj_N', 'Dist_N') to restrict to normal tissue.
    """
    cores, ranking = _resolve_cores(
        bins_cd8a_rev, adata, n_top, min_high_tcell_bins, sort_by, core_ids
    )

    if tissue_filter is not None:
        ranking = ranking[ranking['tissue'].isin(tissue_filter)]
        cores = [c for c in cores if c in set(ranking['core'])]

    if not cores:
        print("No cores to plot")
        return []

    obs = adata.obs
    coords_all = adata.obsm['spatial']
    gdf15_full = _get_gdf15_full(adata, gdf15_gene)
    shared_extent = _compute_shared_extent(cores, obs, coords_all)

    figures = []

    fig_legend = _make_legend_figure(cd8_color, cd3e_color)
    figures.append(('_legend', fig_legend))
    plt.show()

    for core_id in cores:
        fig, axes = plt.subplots(
            1, 3,
            figsize=(figsize_per_panel[0] * 3, figsize_per_panel[1]),
        )

        suptitle = _draw_core_row(
            axes, core_id, ranking, bins_cd8a_rev, bin_geometry,
            obs, coords_all, gdf15_full,
            cd8_color, cd3e_color,
            cd8_size, cd3e_size, epi_size,
            gdf15_gene, gdf15_cmap, show_tcells,
            add_colorbar=True,
            shared_extent=shared_extent,
            col_title_fontsize=col_title_fontsize,
        )

        fig.subplots_adjust(
            wspace=0.05, left=0.04, right=0.96, top=0.90, bottom=0.05,
        )
        fig.suptitle(suptitle, fontsize=11, fontweight='bold', y=0.98)

        figures.append((core_id, fig))
        plt.show()

    return figures


def plot_top_isolated_cores_3x3_grid(
    bins_cd8a_rev, adata, bin_geometry,
    core_ids,
    sort_by='n_LH',
    cd8_color="#FF0066",
    cd3e_color='#00C3FF',
    cd8_size=50, cd3e_size=20, epi_size=10,
    figsize_per_panel=(5, 5),
    gdf15_gene=GDF15_GENE,
    gdf15_cmap='Greens',
    show_tcells=True,
    col_title_fontsize=16,
    row_label_fontsize=18,
):
    """Combine 3 cores into a single 3x3 figure.

    Each row is one core: H&E | LISA bins | GDF15 + T cells.
    Rows are sorted by tissue type: N -> TA -> CA.
    Column titles only appear on the top row.
    Expects exactly 3 core_ids.

    Returns
    -------
    fig : matplotlib.figure.Figure
    fig_legend : matplotlib.figure.Figure
    """
    if len(core_ids) != 3:
        raise ValueError(f"Expected 3 core_ids, got {len(core_ids)}")

    cores, ranking = _resolve_cores(
        bins_cd8a_rev, adata, n_top=None,
        min_high_tcell_bins=0, sort_by=sort_by, core_ids=core_ids,
    )
    if len(cores) != 3:
        raise ValueError(
            f"Only {len(cores)} of the requested cores were found. "
            f"Cannot build a 3x3 grid."
        )

    core_to_tissue = {
        c: ranking[ranking['core'] == c].iloc[0]['tissue']
        for c in cores
    }
    cores = sorted(cores, key=lambda c: _tissue_sort_key(core_to_tissue[c]))

    obs = adata.obs
    coords_all = adata.obsm['spatial']
    gdf15_full = _get_gdf15_full(adata, gdf15_gene)
    shared_extent = _compute_shared_extent(cores, obs, coords_all)

    fig, axes = plt.subplots(
        3, 3,
        figsize=(figsize_per_panel[0] * 3, figsize_per_panel[1] * 3),
        squeeze=False,
    )

    for row_idx, core_id in enumerate(cores):
        suptitle = _draw_core_row(
            axes[row_idx], core_id, ranking, bins_cd8a_rev, bin_geometry,
            obs, coords_all, gdf15_full,
            cd8_color, cd3e_color,
            cd8_size, cd3e_size, epi_size,
            gdf15_gene, gdf15_cmap, show_tcells,
            add_colorbar=True,
            shared_extent=shared_extent,
            show_titles=(row_idx == 0),
            col_title_fontsize=col_title_fontsize,
        )
        axes[row_idx, 0].set_ylabel(
            suptitle, fontsize=row_label_fontsize, fontweight='bold',
            rotation=90, labelpad=10,
        )

    fig.subplots_adjust(
        wspace=-0.1, hspace=0.05,
        left=0.04, right=0.98, top=0.97, bottom=0.02,
    )

    fig_legend = _make_legend_figure(cd8_color, cd3e_color)

    return fig, fig_legend


fig, fig_legend = plot_top_isolated_cores_3x3_grid(
    bins_cd8a_rev, adata, bin_geometry_rev,
    core_ids=['T1-CA-1', 'N2-N-2', 'N1-TA-2'],
    sort_by='n_LH',
    gdf15_cmap='YlGn',
)

fig.savefig(
    f'{output_dir}/top_isolated_cores_3x3.pdf',
    dpi=300, bbox_inches='tight',
)

fig_legend.savefig(
    f'{output_dir}/top_isolated_cores_3x3_legend.pdf',
    dpi=300, bbox_inches='tight',
)

fig_props_cd8a_rev_merged.savefig(
    f'{output_dir}/cd8a_gdf15_quadrant_proportions.pdf',
    dpi=300, bbox_inches='tight',
)

# scatterplots

In [ ]:
from scipy.spatial import cKDTree


def collapse_tissues(metrics_df, mapping=None):
    if mapping is None:
        mapping = {'Dist_N': 'All N', 'Adj_N': 'All N'}
    df = metrics_df.copy()
    df['tissue'] = df['tissue'].replace(mapping)
    return df


def build_core_metrics(
    adata,
    gdf15_gene=GDF15_GENE,
    gdf15_layer='normalized',
    tissue_col=TISSUE_COL,
    patient_col=PATIENT_COL,
    min_cd8=10,
    min_epi=10,
):
    obs = adata.obs
    coords_all = adata.obsm['spatial']
    core_arr = obs[CORE_COL].values
    ct_arr   = obs[CELLTYPE_COL].values

    if gdf15_gene not in adata.var_names:
        raise ValueError(f"{gdf15_gene} not in adata.var_names")
    if gdf15_layer not in adata.layers:
        raise ValueError(f"adata.layers['{gdf15_layer}'] not found")

    g = adata[:, gdf15_gene].layers[gdf15_layer]
    gdf15 = g.toarray().flatten() if hasattr(g, 'toarray') else np.asarray(g).flatten()

    is_epi = ct_arr == EPITHELIAL_LABEL
    is_cd8 = ct_arr == CD8_LABEL

    rows = []
    for core_id in pd.unique(core_arr):
        sel = core_arr == core_id
        epi_sel = sel & is_epi
        cd8_sel = sel & is_cd8

        epi_xy = coords_all[epi_sel]
        cd8_xy = coords_all[cd8_sel]
        n_epi = len(epi_xy)
        n_cd8 = len(cd8_xy)

        if tissue_col in obs.columns and n_epi:
            epi_tissue_vals = obs.loc[epi_sel, tissue_col].unique()
            if len(epi_tissue_vals) == 1:
                tissue = epi_tissue_vals[0]
            else:
                tissue = 'Mixed'
        else:
            tissue = 'Unknown'

        core_obs = obs[sel]
        patient = (
            core_obs[patient_col].iloc[0]
            if patient_col in core_obs.columns and len(core_obs) else 'Unknown'
        )

        mean_gdf15_epi = float(gdf15[epi_sel].mean()) if n_epi else np.nan
        median_gdf15_epi = float(np.median(gdf15[epi_sel])) if n_epi else np.nan

        if n_cd8 >= min_cd8 and n_epi >= min_epi:
            tree_cd8 = cKDTree(cd8_xy)
            tree_epi = cKDTree(epi_xy)
            d_epi_to_cd8, _ = tree_cd8.query(epi_xy, k=1)
            d_cd8_to_epi, _ = tree_epi.query(cd8_xy, k=1)
            median_epi_to_cd8_all = float(np.median(d_epi_to_cd8))
            median_cd8_to_epi_all = float(np.median(d_cd8_to_epi))
        else:
            median_epi_to_cd8_all = np.nan
            median_cd8_to_epi_all = np.nan

        rows.append({
            'core': core_id,
            'tissue': tissue,
            'patient': patient,
            'n_cells': int(sel.sum()),
            'n_epi': n_epi,
            'n_cd8': n_cd8,
            'mean_gdf15_epi': mean_gdf15_epi,
            'median_gdf15_epi': median_gdf15_epi,
            'median_epi_to_cd8_all': median_epi_to_cd8_all,
            'median_cd8_to_epi_all': median_cd8_to_epi_all,
        })

    metrics_df = pd.DataFrame(rows)
    print(f"Built metrics for {len(metrics_df)} cores")
    print(f"  With distance metric: "
          f"{metrics_df['median_epi_to_cd8_all'].notna().sum()}/{len(metrics_df)}")
    return metrics_df


metrics_df = build_core_metrics(adata)

metrics_df = collapse_tissues(metrics_df)
print("\nCores per tissue after collapsing Dist_N → Adj_N:")
print(metrics_df['tissue'].value_counts())


In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def r2_marginal(model):
    """Nakagawa & Schielzeth marginal R² (fixed effects only)."""
    fe_vals = np.dot(model.model.exog, model.fe_params)
    var_fe = np.var(fe_vals)
    var_random = float(model.cov_re.iloc[0, 0])
    var_resid = model.scale
    return var_fe / (var_fe + var_random + var_resid)


def fit_rank_mixed_model_per_tissue(
    metrics_df,
    x_metric='mean_gdf15_epi',
    y_metric='median_epi_to_cd8_all',
    cd8_col='n_cd8',
    tissue_order=('Adj_N', 'AD', 'CA'),
    drop_tissues=('Mixed', 'Unknown', '?', 'nan'),
    collapse_separate_n=True,
    min_patients=4,
):
    df = collapse_tissues(metrics_df) if collapse_separate_n else metrics_df.copy()
    df = df[~df['tissue'].isin(drop_tissues)]
    df = df.dropna(subset=[x_metric, y_metric, 'patient', 'tissue', cd8_col]).copy()

    results = []
    for tissue in tissue_order:
        sub = df[df['tissue'] == tissue].copy()
        n_cores = len(sub)
        n_patients = sub['patient'].nunique()

        if n_patients < min_patients:
            results.append({
                'tissue': tissue,
                'n_cores': n_cores,
                'n_patients': n_patients,
                'spearman_rho': np.nan,
                'slope_rank': np.nan,
                'se_rank': np.nan,
                'p_value': np.nan,
                'adj_p_value': np.nan,
                'adj_cd8_p_value': np.nan,
                'note': 'too few patients',
            })
            continue

        rho, _ = spearmanr(sub[x_metric], sub[y_metric])

        sub['x_rank'] = sub[x_metric].rank()
        sub['y_rank'] = sub[y_metric].rank()
        sub['cd8_rank'] = sub[cd8_col].rank()

        try:
            model = smf.mixedlm(
                'y_rank ~ x_rank',
                data=sub,
                groups=sub['patient'],
            ).fit(method='powell', reml=True)

            try:
                model_adj = smf.mixedlm(
                    'y_rank ~ x_rank + cd8_rank',
                    data=sub,
                    groups=sub['patient'],
                ).fit(method='powell', reml=True)
                adj_p = model_adj.pvalues['x_rank']
                adj_cd8_p = model_adj.pvalues['cd8_rank']
            except Exception:
                adj_p = np.nan
                adj_cd8_p = np.nan

            results.append({
                'tissue': tissue,
                'n_cores': n_cores,
                'n_patients': n_patients,
                'spearman_rho': rho,
                'slope_rank': model.params['x_rank'],
                'se_rank': model.bse['x_rank'],
                'p_value': model.pvalues['x_rank'],
                'adj_p_value': adj_p,
                'adj_cd8_p_value': adj_cd8_p,
                'note': '',
            })
        except Exception as e:
            results.append({
                'tissue': tissue,
                'n_cores': n_cores,
                'n_patients': n_patients,
                'spearman_rho': rho,
                'slope_rank': np.nan,
                'se_rank': np.nan,
                'p_value': np.nan,
                'adj_p_value': np.nan,
                'adj_cd8_p_value': np.nan,
                'note': f'fit failed: {type(e).__name__}',
            })

    return pd.DataFrame(results)


def plot_gdf15_vs_distance_by_tissue_mixed(
    metrics_df,
    x_metric='mean_gdf15_epi',
    x_label='Mean GDF15 / epi cell',
    y_metric='median_epi_to_cd8_all',
    y_label='Median all-epi → nearest CD8 (µm)',
    cd8_col='n_cd8',
    tissue_palette=None,
    tissue_order=('All N', 'AD', 'CA'),
    drop_tissues=('Mixed', 'Unknown', '?', 'nan'),
    figsize_per_panel=(4.0, 4.5),
    point_size=80,
    show_regression=True,
    collapse_separate_n=True,
):
    if tissue_palette is None:
        tissue_palette = {
            'All N':  '#377EB8',
            'AD': '#FF7F00',
            'CA': '#E41A1C',
        }

    df = collapse_tissues(metrics_df) if collapse_separate_n else metrics_df.copy()
    df = df[~df['tissue'].isin(drop_tissues)]
    df = df.dropna(subset=[x_metric, y_metric, 'patient', 'tissue'])

    if df.empty:
        print("Nothing to plot")
        return None

    stats_df = fit_rank_mixed_model_per_tissue(
        metrics_df,
        x_metric=x_metric,
        y_metric=y_metric,
        cd8_col=cd8_col,
        tissue_order=tissue_order,
        drop_tissues=drop_tissues,
        collapse_separate_n=collapse_separate_n,
    )
    stats_lookup = stats_df.set_index('tissue').to_dict('index')

    tissues_present = [t for t in tissue_order if t in df['tissue'].unique()]
    n = len(tissues_present)

    fig, axes = plt.subplots(
        1, n,
        figsize=(figsize_per_panel[0] * n, figsize_per_panel[1]),
        squeeze=False,
    )
    axes = axes[0]

    for ax, t in zip(axes, tissues_present):
        sub = df[df['tissue'] == t]
        ax.scatter(
            sub[x_metric], sub[y_metric],
            c=tissue_palette.get(t, 'gray'),
            s=point_size, alpha=0.85,
            edgecolors='white', linewidths=0.4,
        )

        if show_regression and len(sub) >= 2:
            x_vals = sub[x_metric].values
            y_vals = sub[y_metric].values
            slope, intercept = np.polyfit(x_vals, y_vals, 1)
            x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
            y_line = slope * x_line + intercept
            ax.plot(
                x_line, y_line,
                color=tissue_palette.get(t, 'gray'),
                linewidth=1.5, alpha=0.7, zorder=2,
            )

        s = stats_lookup.get(t, {})
        p = s.get('p_value', np.nan)
        rho = s.get('spearman_rho', np.nan)
        n_cores = s.get('n_cores', len(sub))
        n_patients = s.get('n_patients', sub['patient'].nunique())
        note = s.get('note', '')
        adj_p = s.get('adj_p_value', np.nan)

        if note:
            stat_text = f'{n_cores} cores, {n_patients} patients\n({note})'
        elif np.isfinite(p):
            stat_lines = []
            if np.isfinite(rho):
                stat_lines.append(f'ρ = {rho:.2f},  p = {p:.2g}')
            else:
                stat_lines.append(f'p = {p:.2g}')
            if t == 'CA' and np.isfinite(adj_p):
                stat_lines.append(f'adj. p = {adj_p:.2g} (CD8 covariate)')
            stat_lines.append(f'{n_cores} cores, {n_patients} patients')
            stat_text = '\n'.join(stat_lines)
        else:
            stat_text = f'{n_cores} cores, {n_patients} patients'

        ax.set_title(
            f'{t}\n{stat_text}',
            fontsize=10, fontweight='bold',
            color=tissue_palette.get(t, 'black'),
        )
        ax.set_xlabel(x_label, fontsize=10)
        for sp in ('top', 'right'):
            ax.spines[sp].set_visible(False)

    axes[0].set_ylabel(y_label, fontsize=10)
    plt.tight_layout()
    return fig, stats_df


# ─── Run it ────────────────────────────────────────────────────────────────
fig_mixed, stats_df = plot_gdf15_vs_distance_by_tissue_mixed(
    metrics_df,
    show_regression=True,
)
print("\nPer-tissue mixed-model results (rank-transformed):")
print(stats_df.to_string(index=False))

fig_mixed.savefig(
    f'{output_dir}/gdf15_vs_distance_by_tissue_mixed_model.pdf',
    dpi=300, bbox_inches='tight',
)
plt.show()


In [ ]:
df = collapse_tissues(metrics_df)
sub = df[df['tissue'] == 'All N'].dropna(subset=['mean_gdf15_epi', 'median_epi_to_cd8_all']).copy()
print(sub.groupby('patient').size().sort_values())

sub['x_rank'] = sub['mean_gdf15_epi'].rank()
sub['y_rank'] = sub['median_epi_to_cd8_all'].rank()

model = smf.mixedlm(
    'y_rank ~ x_rank',
    data=sub,
    groups=sub['patient'],
).fit(method='powell', reml=True)
print(model.summary())

In [ ]:
metrics_df.groupby('patient').size()

In [ ]:
metrics_df['mean_gdf15_epi'].duplicated().sum()

In [ ]:

cd8_count_col = 'n_cd8'

df = collapse_tissues(metrics_df).copy()
df = df[~df['tissue'].isin(('Mixed', 'Unknown', '?', 'nan'))]
df = df.dropna(subset=['mean_gdf15_epi', 'median_epi_to_cd8_all', 'patient', 'tissue', cd8_count_col])

ca = df[df['tissue'] == 'CA'].copy()
ca['x_rank'] = ca['mean_gdf15_epi'].rank()
ca['y_rank'] = ca['median_epi_to_cd8_all'].rank()
ca['cd8_rank'] = ca[cd8_count_col].rank()

m_unadj = smf.mixedlm(
    'y_rank ~ x_rank', data=ca, groups=ca['patient'],
).fit(method='powell', reml=True)

m_adj = smf.mixedlm(
    'y_rank ~ x_rank + cd8_rank', data=ca, groups=ca['patient'],
).fit(method='powell', reml=True)

fig, ax = plt.subplots(figsize=(4.5, 4.5))

color = '#E41A1C'
ax.scatter(
    ca['mean_gdf15_epi'], ca['median_epi_to_cd8_all'],
    c=color, s=80, alpha=0.85, edgecolors='white', linewidths=0.4,
)
slope, intercept = np.polyfit(ca['mean_gdf15_epi'], ca['median_epi_to_cd8_all'], 1)
x_line = np.linspace(ca['mean_gdf15_epi'].min(), ca['mean_gdf15_epi'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.5, alpha=0.7)

n_cores = len(ca)
n_patients = ca['patient'].nunique()
ax.set_title(
    f'CA\n'
    f'unadj p = {m_unadj.pvalues["x_rank"]:.3g}  |  '
    f'adj p = {m_adj.pvalues["x_rank"]:.3g}\n'
    f'(CD8 count covariate p = {m_adj.pvalues["cd8_rank"]:.3g})\n'
    f'{n_cores} cores, {n_patients} patients',
    fontsize=9, fontweight='bold', color=color,
)
ax.set_xlabel('Mean GDF15 / epi cell', fontsize=10)
ax.set_ylabel('Median all-epi → nearest CD8 (µm)', fontsize=10)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)

plt.tight_layout()
fig.savefig(f'{output_dir}/confound_check_CA_gdf15_vs_distance.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"Unadjusted: GDF15 slope={m_unadj.params['x_rank']:.4f}, p={m_unadj.pvalues['x_rank']:.4g}")
print(f"Adjusted:   GDF15 slope={m_adj.params['x_rank']:.4f}, p={m_adj.pvalues['x_rank']:.4g}")
print(f"            CD8   slope={m_adj.params['cd8_rank']:.4f}, p={m_adj.pvalues['cd8_rank']:.4g}")

In [ ]:
from scipy.stats import spearmanr

tissue_palette = {'All N': '#377EB8', 'AD': '#FF7F00', 'CA': '#E41A1C', 'Mixed': '#999999'}
tissue_order = ('All N', 'AD', 'CA', 'Mixed')
drop_tissues = ('Unknown', '?', 'nan', 'Dist_N')

x_metric = 'mean_gdf15_epi'
y_metric = 'median_epi_to_cd8_all'

df = collapse_tissues(metrics_df).copy()
df = df[~df['tissue'].isin(drop_tissues)]
df = df.dropna(subset=[x_metric, y_metric])

fig, ax = plt.subplots(figsize=(4, 4))

tissues_present = [t for t in tissue_order if t in df['tissue'].unique()]
for t in tissues_present:
    s = df[df['tissue'] == t]
    ax.scatter(s[x_metric], s[y_metric],
               c=tissue_palette.get(t, 'gray'),
               s=40, alpha=0.8,
               edgecolors='white', linewidths=0.4,
               label=f'{t} (n={len(s)})')

rho, p = spearmanr(df[x_metric], df[y_metric])
ax.text(
    0.97, 0.97, f'ρ = {rho:.2f}\np = {p:.2g}\nn = {len(df)}',
    transform=ax.transAxes, ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor='lightgray', alpha=0.85),
)

ax.set_xlabel('Mean GDF15 / epi cell', fontsize=11)
ax.set_ylabel('Median all-epi → nearest CD8 (µm)', fontsize=11)
ax.set_title('GDF15 burden vs. CD8 proximity', fontsize=12, fontweight='bold')
ax.legend(frameon=False, fontsize=9)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)

plt.tight_layout()
fig.savefig(f'{output_dir}/gdf15_vs_median_epi_to_cd8_all.pdf', dpi=300, bbox_inches='tight')
plt.show()

# T Cell clustering

In [ ]:
t_cell_adata = sc.read_h5ad(str(P.processed.adata.tcells / "tcells_30_50_harmony_05_pt_only.h5ad"))
tcellmarkers_plot = ["CD8A", "GZMA", "GZMB", "CCL4", "IFNG"]

In [ ]:
cluster_colors = {
    "0": '#009432',
    "1": '#C4E538',
    "2": "#FF788F",
    "3": "#85D5FB",
    "4": "#B3A1F9",
    "5": '#0652DD',
    "6": '#F79F1F',
    "7": "#a53707",
    "8": '#833471',
    "9": '#EA2027',
    "10": '#1B1464'
}


tissue_colors = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}



TISSUE_ORDER    = ['Dist_N','Adj_N', 'AD', 'CA', 'NA']
DYSPLASIA_ORDER = ['Dist_N','Adj_N', 'LGD', 'HGD', 'CA', 'NA']


def plot_patient_distribution_by_cluster(adata, cluster_col, figsize=(12, 6)):
    clusters = sorted(adata.obs[cluster_col].unique())
    patients = sorted(adata.obs['patient_id'].unique())
    patient_colors = dict(zip(patients, sns.color_palette("tab20", len(patients))))
    composition = pd.crosstab(
        adata.obs[cluster_col], adata.obs['patient_id'], normalize='index'
    ) * 100
    composition = composition.reindex(columns=patients, fill_value=0).loc[clusters]

    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(clusters))
    for patient in patients:
        ax.bar(range(len(clusters)), composition[patient], bottom=bottom,
               color=patient_colors[patient], label=patient, alpha=0.8)
        bottom += composition[patient].values
    ax.set_ylabel('Composition (%)', fontsize=12, fontweight='bold')
    ax.set_title('Patient Distribution by Cluster', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
    ax.set_ylim(0, 100)
    ax.set_xlabel(cluster_col, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(clusters)))
    ax.set_xticklabels(clusters, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    return fig, ax, composition

def get_filtered_palette(adata, col, palette):
    """Return palette dict filtered to only categories actually present in the data."""
    # Use value_counts to get only categories with cells, not all categorical levels
    present = set(adata.obs[col].astype(str).value_counts()[lambda x: x > 0].index)
    return {k: v for k, v in palette.items() if k in present}

def explore_subclusters(
    cluster_adata,
    key,
    minres=None,
    maxres=None,
    cluster_palette=None,
    dotplots=False,
    ngenes=5,
    show_axes=True,
    umap_title=None,
    point_size=5,
    transcript_markers=None
):
    obs = cluster_adata.obs.columns.tolist()
    reses = [col for col in obs if col.startswith(key)]
    if minres is not None and maxres is None:
        reses = [r for r in reses if float(r[-3:]) >= minres]
    elif minres is not None and maxres is not None:
        reses = [r for r in reses if minres <= float(r[-3:]) <= maxres]
    elif minres is None and maxres is not None:
        reses = [r for r in reses if float(r[-3:]) <= maxres]
    for res in reses:
        color_key = f"{res}_colors"
        if color_key in cluster_adata.uns:
            del cluster_adata.uns[color_key]

    def _fix_axes(axes_obj):
        if not show_axes:
            for a in (axes_obj if isinstance(axes_obj, list) else [axes_obj]):
                a.set_axis_off()
    if transcript_markers and reses:
        print(f"transcript_markers passed in: {transcript_markers}")
        print(f"sample var_names: {cluster_adata.var_names[:10].tolist()}")
        valid_markers = [g for g in transcript_markers if g in cluster_adata.var_names]
        print(f"valid_markers after filtering: {valid_markers}")
    if reses:
        ax = sc.pl.umap(cluster_adata, color=reses, ncols=3, size=point_size,
                        palette=cluster_palette, show=False,
                        title=[umap_title or r for r in reses])
        _fix_axes(ax if isinstance(ax, list) else [ax])
        plt.show()

    # ax = sc.pl.umap(cluster_adata, color=["core_id"], size=point_size, show=False,
    #                 title=umap_title or "core_id")
    # _fix_axes(ax)
    # plt.show()

    # ax = sc.pl.umap(cluster_adata, color=["patient_id"], size=point_size, show=False,
    #                 title=umap_title or "patient_id")
    # _fix_axes(ax)
    # plt.show()

    # for res in reses:
    #     plot_patient_distribution_by_cluster(cluster_adata, res)

    # if dotplots:
    #     for res in reses:
    #         sc.tl.rank_genes_groups(cluster_adata, res, layer='normalized', use_raw=False)
    #         sc.pl.rank_genes_groups_dotplot(cluster_adata, n_genes=ngenes, groupby=res,
    #                                         title=f"{res}", standard_scale="var")
    #         if transcript_markers:
    #             sc.pl.dotplot(cluster_adata, transcript_markers, groupby=res,
    #                           title=res, standard_scale="var")

    # if transcript_markers is not None and reses:
    #     for res in reses:
    #         sc.pl.matrixplot(
    #             cluster_adata,
    #             transcript_markers,
    #             res,
    #             dendrogram=True,
    #             layer='normalized',
    #             use_raw=False,
    #             standard_scale='var',
    #             cmap="Reds",
    #             title=res,
    #         )

In [ ]:
colors_dutch = ['#F79F1F',
 '#1289A7',
 '#009432',
 '#9980FA',
 '#EA2027',
 '#833471',
 '#1B1464']

colors_dutch_y = ['#F79F1F',
 '#A3CB38',
 '#1289A7',
 '#9980FA',
 '#ED4C67',
 '#833471',
 '#D980FA',
 '#009432',
 '#0652DD',
 '#EA2027',
'#1B1464',
 '#EE5A24']

colors_dutch_long = [
    '#FFC312', '#C4E538', '#12CBC4', '#FDA7DF', '#ED4C67',
    '#F79F1F', '#A3CB38', '#1289A7', '#D980FA', '#B53471',
    '#EE5A24', '#009432', '#0652DD', '#9980FA', '#833471',
    '#EA2027', '#006266', '#1B1464', '#5758BB', '#6F1E51'
]

In [ ]:
colors_dutch_y = ['#F79F1F',
 '#ED4C67',
 '#1289A7',
 '#9980FA',
 '#A3CB38',
 '#833471',
 '#D980FA',
 '#009432',
 '#0652DD',
 '#EA2027',
'#1B1464',
 '#EE5A24']

In [ ]:
import matplotlib.pyplot as plt

res = "T-Cells_0.6"
sc.settings._vector_friendly = True

_orig_show = plt.show
plt.show = lambda *a, **kw: None
try:
    explore_subclusters(
        cluster_adata=t_cell_adata,
        key='T-Cells',
        minres=0.6,
        maxres=0.6,
        cluster_palette=colors_dutch_y,
        show_axes=False,
        point_size=5,
        dotplots=True,
        ngenes=15,
        transcript_markers=tcellmarkers_plot,
    )
    for i, num in enumerate(plt.get_fignums()):
        plt.figure(num).savefig(
            f'{output_dir}/explore_subclusters_{res}_{i:02d}.pdf',
            dpi=300, bbox_inches='tight',
        )
finally:
    plt.show = _orig_show
    sc.settings._vector_friendly = False

for num in plt.get_fignums():
    plt.figure(num)
    _orig_show()

In [ ]:
sc.settings._vector_friendly = True
ax = sc.pl.umap(
    t_cell_adata, color="CD8A", cmap="RdYlBu_r",
    vmax="p99", vmin="p1", size=5, show=False, title="CD8A",
)
ax.set_axis_off()

ax.figure.savefig(
    f'{output_dir}/umap_cd8a.pdf',
    dpi=300, bbox_inches='tight',
)

plt.show()
sc.settings._vector_friendly = False

In [ ]:
import matplotlib.pyplot as plt

res = "T-Cells_0.6"
sc.settings._vector_friendly = True

merge_map = {
    '0': 'Other T Cells',
    '1': 'CD8+ T Cells',
    '3': 'Other T Cells',
    '4': 'Other T Cells',
    '2': 'Other T Cells',
}
merged_col = f'{res}_merged'
t_cell_adata.obs[merged_col] = (
    t_cell_adata.obs[res].astype(str).map(merge_map).astype('category')
)
t_cell_adata.obs[merged_col] = t_cell_adata.obs[merged_col].cat.reorder_categories(
    ['CD8+ T Cells', 'Other T Cells']
)

merged_palette = {
    'CD8+ T Cells':  '#FF0066',
    'Other T Cells': '#00C3FF',
}

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.umap(
    t_cell_adata, color=merged_col,
    palette=merged_palette,
    size=8, show=False, ax=ax_left,
    title='T cell subsets',
)
ax_left.set_axis_off()

sc.pl.umap(
    t_cell_adata, color='CD8A', cmap='RdYlBu_r',
    vmax='p99', vmin='p1', size=5, show=False, ax=ax_right,
    title='CD8A',
)
ax_right.set_axis_off()

fig.tight_layout()
fig.savefig(
    f'{output_dir}/umap_clusters_and_cd8a.pdf',
    dpi=300, bbox_inches='tight',
)
plt.show()
sc.settings._vector_friendly = False

In [ ]:
markers = ["CD8A", "CD8B", "GZMA", "GZMB", "CCL4", "PRF1"]


res = "T-Cells_0.6"
merged_col = f'{res}_merged'
if merged_col not in t_cell_adata.obs.columns:
    merge_map = {
    '0': 'Other T Cells',
    '1': 'CD8+ T Cells',
    '3': 'Other T Cells',
    '4': 'Other T Cells',
    '2': 'Other T Cells',
    }
    t_cell_adata.obs[merged_col] = (
        t_cell_adata.obs[res].astype(str).map(merge_map).astype('category')
    )
    t_cell_adata.obs[merged_col] = t_cell_adata.obs[merged_col].cat.reorder_categories(
        ['CD8+ T Cells', 'Other T Cells']
    )

dp = sc.pl.dotplot(
    t_cell_adata,
    var_names=markers,
    groupby=merged_col,
    standard_scale="var",
    return_fig=True,
)
dp.style(dot_edge_color="none", cmap="Reds")
ax_dict = dp.show(return_axes=True)

ax_dict["mainplot_ax"].tick_params(axis="both", labelsize=14)
plt.setp(
    ax_dict["mainplot_ax"].get_xticklabels(),
    rotation=45, ha="right", fontsize=14,
)

for ax_name, ax in ax_dict.items():
    if ax_name != "mainplot_ax":
        ax.tick_params(labelsize=12)
        if ax.get_title():
            ax.set_title(ax.get_title(), fontsize=12)

ax_dict["mainplot_ax"].figure.savefig(
    f'{output_dir}/dotplot_tcell_markers.pdf',
    dpi=300, bbox_inches='tight',
)

plt.show()